# Visualize Preprocessed 2D DNS Snapshots (Static Plots)

This notebook loads a subsampled 2D `.npz` file and provides tools to visualize the data as spatial snapshots or as time-evolution plots along a specific axis.

**Key Features:**
- **Consistent Color Scales:** A single, global color palette is calculated and applied to all plots for accurate visual comparison.
- **Grouped Parameters:** You only need to set parameters once for each group of plots.
- **Individual Execution:** You can run a single cell to regenerate a specific plot after changing the shared parameters.

### 1. Setup and Imports

In [1]:
import numpy as np
from pathlib import Path
import logging

# Import all necessary plotting functions from the new 2D core module
from mhd_surrogate_core.plotting.xz import (
    plot_xz_snapshot,
    plot_x_time_evolution,
    plot_z_time_evolution
)

### 2. User Configuration

**Action Required:** Set your parameters in this cell. You can re-run this cell anytime to apply changes without reloading the data.

In [ ]:
# <<< 1. SET DATA FILE PATH >>>
# This is only used when the 'Load Data' cell is run.
data_file_path = Path("/cephfs/users/skowronek/Documents/PhD/nuclear_fusion_cooling/data/preprocessed_dns_output/01-Cold_Runs/01-Re16K_Ha325/T1492_x1151_y1_z127_c2/T1492_x1151_y1_z127_c2.npz")

# <<< 2. DEFINE CHANNEL NAME MAPPING >>>
# Maps data channel names to display names for plots.
channel_map = {
    'vx': 'u',
    'vz': 'w'
}

# <<< 3. SET GLOBAL PLOT SIZING PARAMETERS >>>
global_base_size = 20
global_min_size = 4

# <<< 4. SET UNIT LABEL AND COLORMAP >>>
global_unit_label = ""
global_cmap = "seismic"

# <<< 5. SET PER-COMPONENT COLOR SCALES (OPTIONAL) >>>
global_vmins = {
    'vx': -2.16,
    'vz': -3.0,
}
global_vmaxs = {
    'vx': 3.84,
    'vz': 3.0,
}

# <<< 6. SET PER-COMPONENT COLOR CENTERING (OPTIONAL) >>>
global_vcenters = {
    'vx': 0.84,
    'vz': 0.0
}

print("Configuration set. You can now run the 'Load Data' cell if you haven't already.")

Configuration set. You can now run the 'Load Data' cell if you haven't already.


### 3. Load Data

**Run this cell only once** at the beginning of your session or after changing the `data_file_path` above.

In [3]:
# --- Data Loading and Initialization ---
timeseries_data = None
coords = {}

if 'data_file_path' not in locals() or not data_file_path.exists():
    logging.error(f"ERROR: Data file not found at {globals().get('data_file_path', 'Not Set')}. Please set the correct path in the configuration cell.")
else:
    with np.load(data_file_path, allow_pickle=True) as data:
        timeseries_data = data['timeseries']
        # Squeeze data to 4D if it's 5D with a singleton dimension
        if timeseries_data.ndim == 5 and timeseries_data.shape[2] == 1:
            timeseries_data = np.squeeze(timeseries_data, axis=2)
            print("Squeezed data from 5D to 4D.")
        
        coords = {
            'labels': list(data['labels']),
            'x': data.get('x_coords', np.arange(timeseries_data.shape[1])),
            'z': data.get('z_coords', np.arange(timeseries_data.shape[2])),
        }
        # Assumes data shape is (time, x, z, channel)
        max_time_index = timeseries_data.shape[0] - 1
        max_x_index = timeseries_data.shape[1] - 1
        max_z_index = timeseries_data.shape[2] - 1
    
    print("Data loaded successfully into memory.")
    print(f"Available channels: {coords['labels']}")
    print(f"Max indices -> Time: {max_time_index}, X: {max_x_index}, Z: {max_z_index}")

# Automatically identify velocity components to be plotted based on the channel_map
velocity_components = sorted([c for c in coords.get('labels', []) if c in channel_map])
if not velocity_components:
    logging.warning("Warning: No velocity components matching the channel_map found in the data.")
else:
    print(f"Identified velocity components to plot: {velocity_components}")

2025-09-20 08:42:50,574 - ERROR - ERROR: Data file not found at /path/to/your/2d_data.npz. Please set the correct path in the configuration cell.
2025-09-20 08:42:50,575 - WARNING - Warning: No velocity components matching the channel_map found in the data.


### 4. Calculate Per-Component Color Scales

Run this cell to determine the final color scales for each component, respecting any overrides set in the config.

In [4]:
# --- Calculate the vmin and vmax for each velocity component ---
final_vmins = {}
final_vmaxs = {}

if timeseries_data is not None and velocity_components:
    for vc in velocity_components:
        # Use override if it exists
        vmin = global_vmins.get(vc)
        vmax = global_vmaxs.get(vc)
        
        # If not overridden, calculate from the data
        if vmin is None or vmax is None:
            vc_idx = coords['labels'].index(vc)
            vc_data = timeseries_data[..., vc_idx]
            if vmin is None:
                vmin = vc_data.min()
            if vmax is None:
                vmax = vc_data.max()
            print(f"Auto-calculated color scale for {vc}: [{vmin:.3f}, {vmax:.3f}]")
        else:
             print(f"Using user-defined color scale for {vc}: [{vmin:.3f}, {vmax:.3f}]")
                
        final_vmins[vc] = vmin
        final_vmaxs[vc] = vmax

---

### 5. Spatial Snapshot Plots (2D)

In [5]:
# === Parameters for 2D Snapshots ===
# <<< MODIFY THESE VALUES >>>
plot_time_index = max_time_index // 2
# ---
print(f"Using time-index {plot_time_index} for snapshot plots.")

NameError: name 'max_time_index' is not defined

In [ ]:
# === Plot Snapshot for u (vx) ===
if 'vx' in velocity_components:
    plot_xz_snapshot(
        data_path=None, # Data is in memory
        timeseries_data=timeseries_data,
        coords=coords,
        channel='vx',
        time_index=plot_time_index,
        channel_alias=channel_map.get('vx'),
        vmin=final_vmins.get('vx'),
        vmax=final_vmaxs.get('vx'),
        vcenter=global_vcenters.get('vx'),
        base_size=global_base_size,
        min_size=global_min_size,
        unit_label=global_unit_label,
        cmap=global_cmap
    )

In [ ]:
# === Plot Snapshot for w (vz) ===
if 'vz' in velocity_components:
    plot_xz_snapshot(
        data_path=None,
        timeseries_data=timeseries_data,
        coords=coords,
        channel='vz',
        time_index=plot_time_index,
        channel_alias=channel_map.get('vz'),
        vmin=final_vmins.get('vz'),
        vmax=final_vmaxs.get('vz'),
        vcenter=global_vcenters.get('vz'),
        base_size=global_base_size,
        min_size=global_min_size,
        unit_label=global_unit_label,
        cmap=global_cmap
    )

---

### 6. Time Evolution Plots (1D Line over Time)

#### 6.1 Evolution along Z-axis (at constant X)

In [ ]:
# === Parameters for Time-Z Plots ===
# <<< MODIFY THESE VALUES >>>
plot_x_index_tz = max_x_index // 2
# ---
print(f"Using x-index {plot_x_index_tz} for Time-Z plots.")

In [ ]:
# === Plot Time-Z Evolution for u (vx) ===
if 'vx' in velocity_components:
    plot_z_time_evolution(
        data_path=None,
        timeseries_data=timeseries_data,
        coords=coords,
        channel='vx',
        x_index=plot_x_index_tz,
        channel_alias=channel_map.get('vx'),
        vmin=final_vmins.get('vx'),
        vmax=final_vmaxs.get('vx'),
        vcenter=global_vcenters.get('vx'),
        base_size=global_base_size,
        min_size=global_min_size,
        unit_label=global_unit_label,
        cmap=global_cmap
    )

In [ ]:
# === Plot Time-Z Evolution for w (vz) ===
if 'vz' in velocity_components:
    plot_z_time_evolution(
        data_path=None,
        timeseries_data=timeseries_data,
        coords=coords,
        channel='vz',
        x_index=plot_x_index_tz,
        channel_alias=channel_map.get('vz'),
        vmin=final_vmins.get('vz'),
        vmax=final_vmaxs.get('vz'),
        vcenter=global_vcenters.get('vz'),
        base_size=global_base_size,
        min_size=global_min_size,
        unit_label=global_unit_label,
        cmap=global_cmap
    )

#### 6.2 Evolution along X-axis (at constant Z)

In [ ]:
# === Parameters for Time-X Plots ===
# <<< MODIFY THESE VALUES >>>
plot_z_index_tx = max_z_index // 2
# ---
print(f"Using z-index {plot_z_index_tx} for Time-X plots.")

In [ ]:
# === Plot Time-X Evolution for u (vx) ===
if 'vx' in velocity_components:
    plot_x_time_evolution(
        data_path=None,
        timeseries_data=timeseries_data,
        coords=coords,
        channel='vx',
        z_index=plot_z_index_tx,
        channel_alias=channel_map.get('vx'),
        vmin=final_vmins.get('vx'),
        vmax=final_vmaxs.get('vx'),
        vcenter=global_vcenters.get('vx'),
        base_size=global_base_size,
        min_size=global_min_size,
        unit_label=global_unit_label,
        cmap=global_cmap
    )

In [ ]:
# === Plot Time-X Evolution for w (vz) ===
if 'vz' in velocity_components:
    plot_x_time_evolution(
        data_path=None,
        timeseries_data=timeseries_data,
        coords=coords,
        channel='vz',
        z_index=plot_z_index_tx,
        channel_alias=channel_map.get('vz'),
        vmin=final_vmins.get('vz'),
        vmax=final_vmaxs.get('vz'),
        vcenter=global_vcenters.get('vz'),
        base_size=global_base_size,
        min_size=global_min_size,
        unit_label=global_unit_label,
        cmap=global_cmap
    )